# Modèle BERTweet : Analyse de sentiment

BERTweet (`vinai/bertweet-base`) : BERT pré-entraîné sur 850M de tweets. Troisième approche du projet, après le modèle classique et le deep learning. Objectif : mesurer le gain par rapport au deep learning. Pas de déploiement prévu (trop lourd pour Azure Free Tier).

## 1. Setup

In [ ]:
import mlflow
mlflow.set_tracking_uri("sqlite:///D:/code/Projet7/notebooks/mlflow.db")
mlflow.set_experiment("bertweet")
mlflow.transformers.autolog(disable=True)

2026/05/07 17:21:43 INFO mlflow.tracking.fluent: Experiment with name 'bertweet' does not exist. Creating a new experiment.


In [ ]:
import torch

print(f"PyTorch version : {torch.__version__}")
print(f"CUDA disponible : {torch.cuda.is_available()}")
print(f"Nombre de GPU   : {torch.cuda.device_count()}")
if torch.cuda.is_available():
    print(f"GPU utilisé     : {torch.cuda.get_device_name(0)}")
    print(f"CUDA version    : {torch.version.cuda}")
    x = torch.tensor([1.0]).cuda()
    print(f"Tensor sur      : {x.device}")
else:
    print("Aucun GPU détecté : torch tourne sur CPU")

PyTorch version : 2.6.0+cu124
CUDA disponible : True
Nombre de GPU   : 1
GPU utilisé     : NVIDIA GeForce RTX 2070
CUDA version    : 12.4
Tensor sur      : cuda:0


## 2. Chargement du tokenizer et du modèle

In [ ]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import torch

tokenizer = AutoTokenizer.from_pretrained("vinai/bertweet-base")
model = AutoModelForSequenceClassification.from_pretrained("vinai/bertweet-base", num_labels=2)

[transformers] emoji is not installed, thus not converting emoticons or emojis into text. Install emoji: pip3 install emoji==0.6.0


Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] RobertaForSequenceClassification LOAD REPORT from: vinai/bertweet-base
Key                         | Status     | 
----------------------------+------------+-
lm_head.dense.weight        | UNEXPECTED | 
lm_head.dense.bias          | UNEXPECTED | 
lm_head.layer_norm.weight   | UNEXPECTED | 
lm_head.decoder.bias        | UNEXPECTED | 
roberta.pooler.dense.weight | UNEXPECTED | 
lm_head.bias                | UNEXPECTED | 
lm_head.layer_norm.bias     | UNEXPECTED | 
lm_head.decoder.weight      | UNEXPECTED | 
roberta.pooler.dense.bias   | UNEXPECTED | 
classifier.dense.bias       | MISSING    | 
classifier.out_proj.weight  | MISSING    | 
classifier.out_proj.bias    | MISSING    | 
classifier.dense.weight     | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


## 3. Chargement des données

Données chargées directement depuis les CSV, sans le prétraitement custom de `src/preprocessing.py` : BERTweet a son propre tokenizer entraîné sur des tweets bruts, un nettoyage supplémentaire dégraderait les performances. Sous-ensemble de 200 000 tweets pour limiter le temps d'entraînement.

In [ ]:
import pandas as pd

# Pas de prétraitement custom : BERTweet a son propre tokenizer pour tweets bruts
train_df = pd.read_csv("../data/train.csv")
val_df = pd.read_csv("../data/val.csv")
test_df = pd.read_csv("../data/test.csv")

SUBSET_SIZE = 200_000
train_df = train_df.sample(n=min(SUBSET_SIZE, len(train_df)), random_state=42).reset_index(drop=True)

print(f"Taille train : {len(train_df)}")
print(f"Taille val   : {len(val_df)}")
print(f"Taille test  : {len(test_df)}")
print(f"\nDistribution train :\n{train_df['target'].value_counts()}")

Taille train : 200000
Taille val   : 159248
Taille test  : 159249

Distribution train :
target
1    100026
0     99974
Name: count, dtype: int64


## 4. Dataset PyTorch

In [ ]:
from torch.utils.data import Dataset, DataLoader

class TweetDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_len=128):
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        encoding = self.tokenizer(self.texts[idx], max_length=self.max_len,
            padding="max_length", truncation=True, return_tensors="pt")
        return {"input_ids": encoding["input_ids"].squeeze(),
                "attention_mask": encoding["attention_mask"].squeeze(),
                "label": torch.tensor(self.labels[idx], dtype=torch.long)}


train_dataset = TweetDataset(
    texts=train_df["text"].tolist(),
    labels=train_df["target"].tolist(),
    tokenizer=tokenizer
)

val_dataset = TweetDataset(
    texts=val_df["text"].tolist(),
    labels=val_df["target"].tolist(),
    tokenizer=tokenizer
)

test_dataset = TweetDataset(
    texts=test_df["text"].tolist(),
    labels=test_df["target"].tolist(),
    tokenizer=tokenizer
)

print(f"Exemples dans train_dataset : {len(train_dataset)}")
print(f"Exemple de batch (clés) : {list(train_dataset[0].keys())}")

Exemples dans train_dataset : 200000
Exemple de batch (clés) : ['input_ids', 'attention_mask', 'label']


## 5. Fine-tuning avec HuggingFace Trainer

Métriques (accuracy, F1, précision, rappel) calculées à chaque epoch de validation et loggées dans MLflow.

In [ ]:
import os
from transformers import Trainer, TrainingArguments
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score


def compute_metrics(pred):
    labels = pred.label_ids
    preds = pred.predictions.argmax(-1)
    return {
        "accuracy": accuracy_score(labels, preds),
        "f1": f1_score(labels, preds),
        "precision": precision_score(labels, preds),
        "recall": recall_score(labels, preds),
    }

os.environ["TENSORBOARD_LOGGING_DIR"] = "./logs"

training_args = TrainingArguments(
    output_dir="./bertweet_output",
    num_train_epochs=2,
    per_device_train_batch_size=32,
    per_device_eval_batch_size=64,
    warmup_steps=500,
    weight_decay=0.01,
    logging_steps=100,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
)

with mlflow.start_run(run_name="bertweet"):
    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=train_dataset,
        eval_dataset=val_dataset,
        compute_metrics=compute_metrics,
    )
    trainer.train()

    preds_output = trainer.predict(test_dataset)
    results = preds_output.metrics

    for key, value in results.items():
        mlflow.log_metric(key, value)
    mlflow.log_param("model_name", "vinai/bertweet-base")
    mlflow.log_param("epochs", 2)
    mlflow.log_param("batch_size", 32)

print("Résultats sur le jeu de test :")
for key, value in results.items():
    print(f"  {key}: {value:.4f}" if isinstance(value, float) else f"  {key}: {value}")

2026/05/07 17:22:09 WARNING mlflow.utils.git_utils: Failed to import Git (the Git executable is probably not on your PATH), so Git SHA is not available. Error: Failed to initialize: Bad git executable.
The git executable must be specified in one of the following ways:
    - be included in your $PATH
    - be set via $GIT_PYTHON_GIT_EXECUTABLE
    - explicitly set via git.refresh(<full-path-to-git-executable>)

All git commands will error until this is rectified.

This initial message can be silenced or aggravated in the future by setting the
$GIT_PYTHON_REFRESH environment variable. Use one of the following values:
    - quiet|q|silence|s|silent|none|n|0: for no message or exception
    - warn|w|warning|log|l|1: for a warning message (logging level CRITICAL, displayed by default)
    - error|e|exception|raise|r|2: for a raised exception

Example:
    export GIT_PYTHON_REFRESH=quiet



Epoch,Training Loss,Validation Loss,Accuracy,F1,Precision,Recall
1,0.320297,0.319236,0.866284,0.872802,0.832065,0.917732
2,0.212850,0.319977,0.877907,0.877126,0.882575,0.871743


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Résultats sur le jeu de test :
  test_loss: 0.3203
  test_accuracy: 0.8660
  test_f1: 0.8723
  test_precision: 0.8325
  test_recall: 0.9161
  test_runtime: 876.1240
  test_samples_per_second: 181.7650
  test_steps_per_second: 2.8410
